# Semantic Search & LLM Application

Prototype asisten customer service e-commerce: FAQ knowledge base → Gemini embeddings → FAISS → semantic search → prompt terstruktur → jawaban Gemini.

Alur: **Ingest → Embed → Store → Retrieve → Generate**.

## 1. Import Library & Load Knowledge Base

In [1]:
import os
import time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
load_dotenv(ROOT / '.env')
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

EMBED_MODEL = 'gemini-embedding-001'
LLM_MODEL = 'gemini-3.5-flash-lite'

df = pd.read_csv(ROOT / 'data' / 'ecommerce_faq_knowledge_base.csv')
print('Total entri knowledge base:', len(df))
df.head()

Total entri knowledge base: 59


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


In [2]:
print('Kolom:', list(df.columns))
print('Missing value:', df.isna().sum().sum())
print('faq_id unik:', df['faq_id'].is_unique)
print()
print(df['category'].value_counts().to_string())

Kolom: ['faq_id', 'category', 'question', 'answer']
Missing value: 0
faq_id unik: True

category
Pemesanan           9
Pengiriman          9
Pembayaran          8
Pengembalian        8
Akun                7
Produk              7
Promo               6
Customer Service    5


In [3]:
df['answer_words'] = df['answer'].str.split().str.len()
df['answer_words'].describe()

count    59.000000
mean     17.169492
std       2.600746
min      11.000000
25%      15.000000
50%      17.000000
75%      19.000000
max      23.000000
Name: answer_words, dtype: float64

Knowledge base berisi 59 entri dalam 8 kategori. Jawaban rata-rata pendek (1-3 kalimat), jadi satu entri FAQ sudah cukup kecil untuk menjadi satu unit retrieval tanpa chunking.

## 2. Menyiapkan Unit Retrievable

Question dan answer digabung menjadi satu teks per entri. Kategori ikut disertakan karena memberi sinyal topik tambahan (misal "Pengembalian" untuk query soal refund). Data asli di `data/` tidak diubah; kolom `content` hanya ada di memori.

In [4]:
df['content'] = (
    'Kategori: ' + df['category']
    + '\nPertanyaan: ' + df['question']
    + '\nJawaban: ' + df['answer']
)
documents = df['content'].tolist()
print('Contoh unit retrievable:\n')
print(documents[0])

Contoh unit retrievable:

Kategori: Pemesanan
Pertanyaan: Bagaimana cara melakukan pemesanan di platform ini?
Jawaban: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Model `gemini-embedding-001` (`text-embedding-004` pada starter sudah tidak tersedia). Dokumen di-embed dengan `task_type=RETRIEVAL_DOCUMENT` dan query dengan `RETRIEVAL_QUERY`, sesuai rekomendasi Gemini untuk skenario asymmetric retrieval. Vektor dinormalisasi L2 agar inner product = cosine similarity.

In [5]:
# Free tier Gemini membatasi request per menit; retry dengan jeda saat kena 429/5xx.
def with_retry(fn, retries=4):
    for attempt in range(retries):
        try:
            return fn()
        except genai.errors.APIError as err:
            if attempt == retries - 1 or err.code not in (429, 500, 503):
                raise
            time.sleep(20 * (attempt + 1))


def embed(texts, task_type):
    result = with_retry(lambda: client.models.embed_content(
        model=EMBED_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(task_type=task_type),
    ))
    vectors = np.array([e.values for e in result.embeddings], dtype='float32')
    faiss.normalize_L2(vectors)
    return vectors


embeddings = embed(documents, 'RETRIEVAL_DOCUMENT')
print('Shape embeddings:', embeddings.shape)
print('Norm vektor pertama:', np.linalg.norm(embeddings[0]).round(4))
assert embeddings.shape[0] == len(df)

Shape embeddings: (59, 3072)
Norm vektor pertama: 1.0


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

`IndexFlatIP` (exact search, inner product). Dengan vektor ternormalisasi, skornya adalah cosine similarity: makin tinggi makin relevan. Untuk 59 vektor, exact search sudah instan sehingga index aproksimasi (IVF/HNSW) tidak diperlukan.

In [6]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print('Dimensi:', dimension)
print('Jumlah vektor tersimpan di index:', index.ntotal)
assert index.ntotal == len(df)

index_path = ROOT / 'index' / 'faq_index.faiss'
index_path.parent.mkdir(exist_ok=True)
faiss.write_index(index, str(index_path))

loaded = faiss.read_index(str(index_path))
print('Index tersimpan di', index_path.relative_to(ROOT), '| ntotal setelah load:', loaded.ntotal)

Dimensi: 3072
Jumlah vektor tersimpan di index: 59
Index tersimpan di index\faq_index.faiss | ntotal setelah load: 59


## 5. Mengimplementasikan Semantic Search

In [7]:
def semantic_search(query, top_k=3):
    query_vector = embed([query], 'RETRIEVAL_QUERY')
    scores, indices = index.search(query_vector, top_k)

    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        row = df.iloc[idx]
        results.append({
            'rank': rank,
            'faq_id': int(row['faq_id']),
            'category': row['category'],
            'question': row['question'],
            'answer': row['answer'],
            'score': round(float(score), 4),
        })
    return results


def show(results):
    return pd.DataFrame(results)[['rank', 'faq_id', 'category', 'question', 'score']]


show(semantic_search('kenapa paket saya belum sampai?'))

,rank,faq_id,category,question,score
0,1,12,Pengiriman,Bagaimana jika paket saya belum sampai melebih...,0.7971
1,2,14,Pengiriman,Kenapa status pengiriman saya tidak berubah se...,0.7816
2,3,52,Pengiriman,Kenapa nomor resi saya belum bisa dilacak di w...,0.7593


## 6. Mendesain Prompt dan Context

Prompt dipisah menjadi empat bagian eksplisit:

- **Persona**: agen CS e-commerce yang ramah dan lugas.
- **Task**: jawab hanya dari context, akui jika informasi tidak ada, jangan mengarang kebijakan.
- **Context**: top-k FAQ hasil retrieval beserta `faq_id` dan skor similarity.
- **Format**: Bahasa Indonesia, singkat, langkah bernomor jika prosedural, sebut sumber FAQ.

Skor similarity ikut ditampilkan di context agar LLM tahu bahwa entri dengan skor rendah mungkin tidak relevan dan tidak memaksakan jawaban dari situ.

In [8]:
PROMPT_TEMPLATE = '''# PERSONA
Kamu adalah agen customer service sebuah platform e-commerce. Gaya bicaramu ramah, sopan, dan to the point.

# TASK
Jawab pertanyaan pelanggan HANYA berdasarkan entri FAQ pada bagian CONTEXT.
- Gunakan hanya entri FAQ yang benar-benar menjawab pertanyaan; abaikan entri yang tidak relevan walaupun ikut muncul di CONTEXT.
- Jangan menambahkan kebijakan, angka, nama layanan, atau prosedur yang tidak tertulis di CONTEXT, dan jangan memakai pengetahuan umum di luar CONTEXT.
- Jika CONTEXT tidak memuat jawaban, katakan dengan jelas bahwa informasi tersebut belum tersedia di FAQ, lalu sarankan pelanggan menghubungi customer service.
- Jika CONTEXT hanya menjawab sebagian, jawab bagian yang tersedia dan sebutkan bagian mana yang tidak tercakup.

# CONTEXT
{context}

# FORMAT
- Bahasa Indonesia, maksimal 4 kalimat atau 5 poin.
- Jika jawabannya berupa prosedur, tulis sebagai langkah bernomor.
- Akhiri dengan satu baris: "Sumber: FAQ #<id>" (pisahkan dengan koma jika lebih dari satu), atau "Sumber: -" jika tidak ada FAQ yang relevan.

# PERTANYAAN PELANGGAN
{query}
'''


def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"[FAQ #{r['faq_id']} | {r['category']} | similarity {r['score']:.3f}]\n"
        f"Pertanyaan: {r['question']}\nJawaban: {r['answer']}"
        for r in search_results
    )
    return PROMPT_TEMPLATE.format(context=context, query=query)


print(build_prompt('apakah bisa bayar pakai cicilan?', semantic_search('apakah bisa bayar pakai cicilan?')))

# PERSONA
Kamu adalah agen customer service sebuah platform e-commerce. Gaya bicaramu ramah, sopan, dan to the point.

# TASK
Jawab pertanyaan pelanggan HANYA berdasarkan entri FAQ pada bagian CONTEXT.
- Gunakan hanya entri FAQ yang benar-benar menjawab pertanyaan; abaikan entri yang tidak relevan walaupun ikut muncul di CONTEXT.
- Jangan menambahkan kebijakan, angka, nama layanan, atau prosedur yang tidak tertulis di CONTEXT, dan jangan memakai pengetahuan umum di luar CONTEXT.
- Jika CONTEXT tidak memuat jawaban, katakan dengan jelas bahwa informasi tersebut belum tersedia di FAQ, lalu sarankan pelanggan menghubungi customer service.
- Jika CONTEXT hanya menjawab sebagian, jawab bagian yang tersedia dan sebutkan bagian mana yang tidak tercakup.

# CONTEXT
[FAQ #20 | Pembayaran | similarity 0.807]
Pertanyaan: Apakah saya bisa membayar dengan cicilan?
Jawaban: Fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan pihak bank.

[FAQ #17 

## 7. Mengintegrasikan LLM

`generate_answer` menyatukan seluruh alur: query → retrieval → prompt → jawaban. Temperature rendah (0.2) dipakai karena jawaban CS harus konsisten dan setia pada FAQ, bukan kreatif.

In [9]:
def call_llm(prompt):
    response = with_retry(lambda: client.models.generate_content(
        model=LLM_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.2),
    ))
    return response.text.strip()


def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)
    answer = call_llm(build_prompt(query, search_results))
    return answer, search_results


answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:\n')
print(answer)
show(sources)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Jawaban LLM:

Halo Kak! Ya, fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan pihak bank. 

Sumber: FAQ #20


,rank,faq_id,category,question,score
0,1,20,Pembayaran,Apakah saya bisa membayar dengan cicilan?,0.8075
1,2,17,Pembayaran,Metode pembayaran apa saja yang tersedia?,0.6936
2,3,22,Pembayaran,Apakah bisa membayar menggunakan lebih dari sa...,0.6881


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Tujuh query dibagi tiga kelompok. Kolom `expected` berisi `faq_id` yang seharusnya ter-retrieve (ditentukan manual dengan membaca knowledge base); `None` berarti informasinya memang tidak ada di FAQ.

In [10]:
test_cases = [
    # Ada jawabannya secara langsung
    {'type': 'direct', 'query': 'Bagaimana cara membatalkan pesanan yang sudah dibuat?', 'expected': 3},
    {'type': 'direct', 'query': 'Metode pembayaran apa saja yang tersedia?', 'expected': 17},
    # Parafrase / sinonim / bahasa informal
    {'type': 'paraphrase', 'query': 'aku lupa kata sandi, gimana caranya bisa masuk lagi ke akunku?', 'expected': 32},
    {'type': 'paraphrase', 'query': 'saldo m-banking udah kepotong tapi orderan masih nunggu dibayar, kok gitu?', 'expected': 21},
    {'type': 'paraphrase', 'query': 'barang yang dateng pecah dan penyok, harus ngapain?', 'expected': 16},
    # Tidak ada di knowledge base
    {'type': 'out-of-kb', 'query': 'Apakah ada toko offline di Surabaya untuk ambil barang langsung?', 'expected': None},
    {'type': 'out-of-kb', 'query': 'Bisa kirim paket ke luar negeri, misalnya ke Singapura?', 'expected': None},
]

rows = []
for case in test_cases:
    answer, sources = generate_answer(case['query'])
    retrieved = [s['faq_id'] for s in sources]
    rows.append({
        'type': case['type'],
        'query': case['query'],
        'expected': case['expected'],
        'top3_ids': retrieved,
        'top3_scores': [s['score'] for s in sources],
        'hit@1': None if case['expected'] is None else retrieved[0] == case['expected'],
        'hit@3': None if case['expected'] is None else case['expected'] in retrieved,
        'answer': answer,
    })

    print(f"[{case['type']}] {case['query']}")
    for s in sources:
        print(f"  #{s['faq_id']:>2} ({s['score']:.3f}) {s['question']}")
    print('\nJawaban:\n' + answer)
    print('-' * 90)

results = pd.DataFrame(rows)

[direct] Bagaimana cara membatalkan pesanan yang sudah dibuat?
  # 3 (0.787) Bagaimana cara membatalkan pesanan yang sudah dibuat?
  #24 (0.713) Bagaimana prosedur pengembalian barang yang tidak sesuai?
  #44 (0.701) Bagaimana jika produk yang saya pesan ternyata kehabisan stok?

Jawaban:
Halo! Pesanan dapat dibatalkan melalui halaman 'Pesanan Saya' selama status pesanan masih 'Diproses'. Namun, perlu diingat bahwa pembatalan tidak dapat dilakukan setelah status pesanan berubah menjadi 'Dikirim'.

Sumber: FAQ #3
------------------------------------------------------------------------------------------


[direct] Metode pembayaran apa saja yang tersedia?
  #17 (0.796) Metode pembayaran apa saja yang tersedia?
  #22 (0.726) Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
  #20 (0.706) Apakah saya bisa membayar dengan cicilan?

Jawaban:
Halo! Kami mendukung transfer bank, kartu kredit/debit, e-wallet, dan pembayaran melalui minimarket rekanan.

Sumber: FAQ #17
------------------------------------------------------------------------------------------


[paraphrase] aku lupa kata sandi, gimana caranya bisa masuk lagi ke akunku?
  #32 (0.775) Saya lupa password akun, bagaimana cara mengaturnya ulang?
  #55 (0.713) Kenapa akun saya tiba-tiba tidak bisa login?
  #31 (0.664) Bagaimana cara mendaftar akun baru?

Jawaban:
Halo! Untuk mengatur ulang password Anda, silakan ikuti langkah-langkah berikut:
1. Klik 'Lupa Password' pada halaman login.
2. Masukkan email terdaftar.
3. Ikuti instruksi reset password yang dikirim ke email Anda.

Sumber: FAQ #32
------------------------------------------------------------------------------------------


[paraphrase] saldo m-banking udah kepotong tapi orderan masih nunggu dibayar, kok gitu?
  #21 (0.769) Kenapa pembayaran saya berhasil dipotong tapi status pesanan masih belum terbayar?
  # 8 (0.705) Kenapa pesanan saya otomatis dibatalkan sistem?
  # 4 (0.702) Berapa lama waktu yang dibutuhkan sebelum pesanan diproses penjual?

Jawaban:
Halo! Hal tersebut biasanya terjadi karena adanya keterlambatan sinkronisasi sistem pembayaran. Silakan tunggu selama 15-30 menit atau hubungi customer service jika status pesanan Anda belum juga berubah. 

Sumber: FAQ #21
------------------------------------------------------------------------------------------


[paraphrase] barang yang dateng pecah dan penyok, harus ngapain?
  #16 (0.800) Apa yang harus dilakukan jika paket diterima dalam kondisi rusak?
  #54 (0.743) Saya menerima barang yang salah, apa yang harus saya lakukan?
  #24 (0.728) Bagaimana prosedur pengembalian barang yang tidak sesuai?

Jawaban:
Halo! Jika paket diterima dalam kondisi rusak, silakan segera ambil foto atau video kondisi paket saat diterima. Setelah itu, laporkan melalui menu komplain dalam waktu 2x24 jam sejak barang diterima.

Sumber: FAQ #16
------------------------------------------------------------------------------------------


[out-of-kb] Apakah ada toko offline di Surabaya untuk ambil barang langsung?
  #58 (0.642) Apakah bisa memesan dalam jumlah besar untuk kebutuhan reseller?
  #11 (0.639) Apakah tersedia opsi pengiriman same-day?
  #43 (0.636) Apakah stok yang tertera di aplikasi selalu update secara real-time?

Jawaban:
Halo! Mohon maaf, informasi mengenai ketersediaan toko offline di Surabaya untuk pengambilan barang langsung belum tersedia di FAQ kami. Silakan menghubungi customer service kami untuk mendapatkan bantuan lebih lanjut.

Sumber: -
------------------------------------------------------------------------------------------


[out-of-kb] Bisa kirim paket ke luar negeri, misalnya ke Singapura?
  #15 (0.689) Apakah bisa memilih jasa pengiriman sendiri saat checkout?
  #12 (0.685) Bagaimana jika paket saya belum sampai melebihi estimasi?
  #13 (0.684) Apakah ongkos kirim sudah termasuk asuransi pengiriman?

Jawaban:
Halo! Mohon maaf, informasi mengenai pengiriman paket ke luar negeri seperti ke Singapura belum tersedia di FAQ kami. Silakan hubungi customer service kami untuk mendapatkan bantuan lebih lanjut mengenai hal tersebut.

Sumber: -
------------------------------------------------------------------------------------------


In [11]:
results[['type', 'query', 'expected', 'top3_ids', 'top3_scores', 'hit@1', 'hit@3']]

,type,query,expected,top3_ids,top3_scores,hit@1,hit@3
0,direct,Bagaimana cara membatalkan pesanan yang sudah ...,3.0,"[3, 24, 44]","[0.7866, 0.7134, 0.701]",True,True
1,direct,Metode pembayaran apa saja yang tersedia?,17.0,"[17, 22, 20]","[0.7957, 0.7261, 0.7062]",True,True
2,paraphrase,"aku lupa kata sandi, gimana caranya bisa masuk...",32.0,"[32, 55, 31]","[0.7749, 0.7134, 0.664]",True,True
3,paraphrase,saldo m-banking udah kepotong tapi orderan mas...,21.0,"[21, 8, 4]","[0.7692, 0.7045, 0.7023]",True,True
4,paraphrase,"barang yang dateng pecah dan penyok, harus nga...",16.0,"[16, 54, 24]","[0.7998, 0.7429, 0.7282]",True,True
5,out-of-kb,Apakah ada toko offline di Surabaya untuk ambi...,NaN,"[58, 11, 43]","[0.6418, 0.6392, 0.6358]",None,None
6,out-of-kb,"Bisa kirim paket ke luar negeri, misalnya ke S...",NaN,"[15, 12, 13]","[0.6886, 0.6853, 0.6843]",None,None


In [12]:
in_kb = results[results['expected'].notna()]
print(f"Hit@1 (query in-KB): {in_kb['hit@1'].mean():.0%}")
print(f"Hit@3 (query in-KB): {in_kb['hit@3'].mean():.0%}")
print()
print('Skor top-1 rata-rata per tipe query:')
print(results.assign(top1=results['top3_scores'].str[0]).groupby('type', sort=False)['top1'].agg(['mean', 'min', 'max']).round(3))

Hit@1 (query in-KB): 100%
Hit@3 (query in-KB): 100%

Skor top-1 rata-rata per tipe query:
             mean    min    max
type                           
direct      0.791  0.787  0.796
paraphrase  0.781  0.769  0.800
out-of-kb   0.665  0.642  0.689


In [13]:
results.to_csv(ROOT / 'documentation' / 'retrieval_test_results.csv', index=False)

### Catatan Analisis Relevansi

| # | Tipe | Query | Top-1 (skor) | Retrieval | Jawaban LLM |
|---|---|---|---|---|---|
| 1 | direct | Bagaimana cara membatalkan pesanan yang sudah dibuat? | #3 (0.787) | Tepat | Sesuai FAQ #3, sumber dikutip |
| 2 | direct | Metode pembayaran apa saja yang tersedia? | #17 (0.796) | Tepat | Menyalin isi FAQ #17 tanpa tambahan |
| 3 | paraphrase | aku lupa kata sandi, gimana caranya bisa masuk lagi ke akunku? | #32 (0.775) | Tepat | Diubah jadi 3 langkah bernomor sesuai format |
| 4 | paraphrase | saldo m-banking udah kepotong tapi orderan masih nunggu dibayar, kok gitu? | #21 (0.769) | Tepat | Sesuai FAQ #21 (tunggu 15-30 menit / hubungi CS) |
| 5 | paraphrase | barang yang dateng pecah dan penyok, harus ngapain? | #16 (0.800) | Tepat | Sesuai FAQ #16 (foto/video, lapor 2x24 jam) |
| 6 | out-of-kb | Apakah ada toko offline di Surabaya untuk ambil barang langsung? | #58 (0.642) | Tidak relevan (memang tidak ada) | Menyatakan info belum tersedia, arahkan ke CS, `Sumber: -` |
| 7 | out-of-kb | Bisa kirim paket ke luar negeri, misalnya ke Singapura? | #15 (0.689) | Tidak relevan (memang tidak ada) | Menyatakan info belum tersedia, arahkan ke CS, `Sumber: -` |

Hit@1 dan Hit@3 untuk 5 query in-KB: 100%.

**Retrieval akurat.**
- Kelima query yang jawabannya ada di FAQ mendapat entri yang benar di peringkat 1.
- Query parafrase memakai kata yang tidak ada di FAQ sama sekali, tapi tetap ketemu karena embedding menangkap makna:
  - "kata sandi" / "masuk lagi" → "password" / "mengatur ulang"
  - "saldo m-banking kepotong" / "orderan nunggu dibayar" → "pembayaran dipotong" / "status belum terbayar"
  - "pecah dan penyok" → "kondisi rusak"
- Skor top-1 query parafrase (rata-rata 0.781) hampir sama dengan query langsung (0.791).
- Skor query langsung tidak mendekati 1.0, padahal teksnya identik dengan pertanyaan FAQ. Ini karena unit yang di-embed juga berisi kategori dan jawaban, bukan pertanyaan saja.

**Retrieval kurang relevan.**
- Dua query out-of-KB tetap mengembalikan 3 entri, karena FAISS selalu mengembalikan top-k. Entri-entri itu hanya mirip topik:
  - toko offline → FAQ reseller, same-day, dan stok
  - kirim ke luar negeri → FAQ jasa pengiriman dan asuransi
- Skor top-1-nya (0.642 dan 0.689) jelas lebih rendah daripada semua query in-KB (minimal 0.769).
- Margin antara peringkat 1 dan 3 juga sangat tipis (≤0.006), artinya tidak ada entri yang menonjol.
- Di peringkat 2-3 query in-KB masih ada entri yang hanya mirip topik, misalnya #24 pengembalian untuk query pembatalan. Inilah alasan prompt menyuruh LLM mengabaikan entri yang tidak menjawab.

**Kualitas jawaban LLM.**
- Pada 5 query in-KB, jawaban hanya mengambil dari FAQ peringkat 1, mengikuti format (langkah bernomor untuk prosedur), dan menyebut sumber dengan benar.
- Entri peringkat 2-3 yang tidak relevan tidak ikut dicampur ke jawaban.
- Pada 2 query out-of-KB, LLM tidak mengarang, misalnya tidak menyebut alamat toko atau tarif ongkir internasional. LLM menyatakan informasinya belum tersedia dan mengarahkan ke customer service, sesuai instruksi di bagian TASK.

**Keterbatasan.**
- **Tidak ada threshold skor.** Penolakan untuk query out-of-KB sepenuhnya bergantung pada LLM. Dari data uji ini, threshold sekitar 0.72 sudah memisahkan in-KB (≥0.769) dari out-of-KB (≤0.689) dan bisa menghemat panggilan LLM. Tapi angka ini baru dari 7 query, jadi perlu divalidasi dengan query yang lebih banyak.
- **Jumlah uji kecil.** Hanya 7 query, dan pertanyaan majemuk (dua topik sekaligus) belum diuji.
- **Ketergantungan pada API.** Embedding dan LLM tergantung kuota free tier Gemini. `gemini-3.8-flash` hanya 20 request/hari, sehingga dipakai `gemini-3.5-flash-lite`.
- **Knowledge base kecil.** Hanya 59 entri, banyak pertanyaan umum tidak tercakup (toko offline, pengiriman internasional).